In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path

path = Path("data/Einzelteil/Einzelteil_T01.txt")

with path.open("rb") as f:
    sample = f.read(20_000)

sample = sample.decode("utf-8", errors="replace")

print(sample[:20_000])

"X1" | | "ID_T01.x" | | "Produktionsdatum.x" | | "Herstellernummer.x" | | "Werksnummer.x" | | "Fehlerhaft.x" | | "Fehlerhaft_Datum.x" | | "Fehlerhaft_Fahrleistung.x" | | "ID_T01.y" | | "Produktionsdatum.y" | | "Herstellernummer.y" | | "Werksnummer.y" | | "Fehlerhaft.y" | | "Fehlerhaft_Datum.y" | | "Fehlerhaft_Fahrleistung.y" | | "ID_T01" | | "Produktionsdatum" | | "Herstellernummer" | | "Werksnummer" | | "Fehlerhaft" | | "Fehlerhaft_Datum" | | "Fehlerhaft_Fahrleistung" "1" | | 660 | | "1-201-2011-247" | | 2008-11-07 | | "201" | | 2011 | | 0 | | NA | | 0 | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA "2" | | 661 | | "1-201-2011-429" | | 2008-11-07 | | "201" | | 2011 | | 0 | | NA | | 0 | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA "3" | | 662 | | "1-201-2011-363" | | 2008-11-07 | | "201" | | 2011 | | 1 | | 2009-09-30 | | 12983 | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | NA | | 

In [3]:
import re
import pandas as pd


def read_t01_chunks(
    path="data/Einzelteil/Einzelteil_T01.txt",
    rows_per_chunk=10_000,
    read_size=4 * 1024 * 1024
):
    delimiter = b" | | "
    boundary = re.compile(rb'^(.*?)\s+"(\d+)"$')

    def raw_tokens():
        carry = b""

        with open(path, "rb") as f:
            while True:
                chunk = f.read(read_size)

                if not chunk:
                    break

                parts = (carry + chunk).split(delimiter)
                carry = parts.pop()

                for part in parts:
                    yield part.strip()

            if carry.strip():
                yield carry.strip()

    def tokens_with_rows():
        expected_row = 1

        for token in raw_tokens():
            match = boundary.match(token)

            if match and int(match.group(2)) == expected_row:
                previous_value = match.group(1).strip()

                if previous_value:
                    yield previous_value

                yield ("ROW", expected_row)
                expected_row += 1

            else:
                yield token

    tokens = iter(tokens_with_rows())

    columns = []

    for _ in range(22):
        token = next(tokens)

        if isinstance(token, tuple):
            raise ValueError("Unexpected row boundary while reading header.")

        columns.append(
            token.decode("utf-8").strip('"')
        )

    rows = []

    while True:
        try:
            marker = next(tokens)
        except StopIteration:
            break

        if not isinstance(marker, tuple):
            raise ValueError(
                f"Expected row boundary, got: {marker[:100]!r}"
            )

        row_number = marker[1]
        values = []

        for _ in range(22):
            try:
                token = next(tokens)
            except StopIteration:
                raise ValueError(
                    f"File ended in the middle of row {row_number}"
                )

            if isinstance(token, tuple):
                raise ValueError(
                    f"Row {row_number} ended after only "
                    f"{len(values)} values"
                )

            value = token.decode("utf-8")

            if (
                len(value) >= 2
                and value[0] == '"'
                and value[-1] == '"'
            ):
                value = value[1:-1]

            if value == "NA":
                value = pd.NA

            values.append(value)

        rows.append(values)

        if len(rows) >= rows_per_chunk:
            yield pd.DataFrame(rows, columns=columns)
            rows = []

    if rows:
        yield pd.DataFrame(rows, columns=columns)

In [4]:
chunks = read_t01_chunks()

data_t01_test = next(chunks)

data_t01_test.head()

,X1,ID_T01.x,Produktionsdatum.x,Herstellernummer.x,Werksnummer.x,Fehlerhaft.x,Fehlerhaft_Datum.x,Fehlerhaft_Fahrleistung.x,ID_T01.y,Produktionsdatum.y,...,Fehlerhaft.y,Fehlerhaft_Datum.y,Fehlerhaft_Fahrleistung.y,ID_T01,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,660,1-201-2011-247,2008-11-07,201,2011,0,NaN,0,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,661,1-201-2011-429,2008-11-07,201,2011,0,NaN,0,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,662,1-201-2011-363,2008-11-07,201,2011,1,2009-09-30,12983,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,663,1-201-2011-30,2008-11-07,201,2011,0,NaN,0,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,664,1-201-2011-72,2008-11-07,201,2011,1,2009-09-30,12983,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [5]:
groups = {
    "x": "ID_T01.x",
    "y": "ID_T01.y",
    "base": "ID_T01"
}

counts = {
    "rows": 0,
    "x": 0,
    "y": 0,
    "base": 0,
    "multiple_groups": 0
}

for chunk in read_t01_chunks():
    counts["rows"] += len(chunk)

    present_x = chunk["ID_T01.x"].notna()
    present_y = chunk["ID_T01.y"].notna()
    present_base = chunk["ID_T01"].notna()

    counts["x"] += present_x.sum()
    counts["y"] += present_y.sum()
    counts["base"] += present_base.sum()

    counts["multiple_groups"] += (
        present_x.astype(int)
        + present_y.astype(int)
        + present_base.astype(int)
        > 1
    ).sum()

counts

{'rows': 3204104,
 'x': np.int64(1281642),
 'y': np.int64(1281642),
 'base': np.int64(640820),
 'multiple_groups': np.int64(0)}

In [6]:
import numpy as np
import pandas as pd

def clean_t01_chunk(chunk):
    out = pd.DataFrame()

    out["X1"] = chunk["X1"]

    out["source_group"] = np.select(
        [
            chunk["ID_T01.x"].notna(),
            chunk["ID_T01.y"].notna(),
            chunk["ID_T01"].notna()
        ],
        [
            "x",
            "y",
            "base"
        ],
        default="unknown"
    )

    fields = [
        "ID_T01",
        "Produktionsdatum",
        "Herstellernummer",
        "Werksnummer",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung"
    ]

    for field in fields:
        out[field] = (
            chunk[
                [
                    f"{field}.x",
                    f"{field}.y",
                    field
                ]
            ]
            .bfill(axis=1)
            .iloc[:, 0]
        )

    return out

In [7]:
test_clean = clean_t01_chunk(data_t01_test)

test_clean.head()

,X1,source_group,ID_T01,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,660,x,1-201-2011-247,2008-11-07,201,2011,0,<NA>,0
1,661,x,1-201-2011-429,2008-11-07,201,2011,0,<NA>,0
2,662,x,1-201-2011-363,2008-11-07,201,2011,1,2009-09-30,12983
3,663,x,1-201-2011-30,2008-11-07,201,2011,0,<NA>,0
4,664,x,1-201-2011-72,2008-11-07,201,2011,1,2009-09-30,12983


In [8]:
test_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   X1                       10000 non-null  str   
 1   source_group             10000 non-null  str   
 2   ID_T01                   10000 non-null  object
 3   Produktionsdatum         10000 non-null  object
 4   Herstellernummer         10000 non-null  object
 5   Werksnummer              10000 non-null  object
 6   Fehlerhaft               10000 non-null  object
 7   Fehlerhaft_Datum         2505 non-null   object
 8   Fehlerhaft_Fahrleistung  10000 non-null  object
dtypes: object(7), str(2)
memory usage: 703.3+ KB


In [9]:
def convert_t01_types(df):
    df = df.copy()

    df["X1"] = pd.to_numeric(
        df["X1"],
        errors="coerce"
    ).astype("Int64")

    df["Produktionsdatum"] = pd.to_datetime(
        df["Produktionsdatum"],
        errors="coerce"
    )

    df["Fehlerhaft_Datum"] = pd.to_datetime(
        df["Fehlerhaft_Datum"],
        errors="coerce"
    )

    df["Fehlerhaft"] = pd.to_numeric(
        df["Fehlerhaft"],
        errors="coerce"
    ).astype("Int8")

    df["Fehlerhaft_Fahrleistung"] = pd.to_numeric(
        df["Fehlerhaft_Fahrleistung"],
        errors="coerce"
    ).astype("Float64")

    return df

In [10]:
test_clean = convert_t01_types(test_clean)

test_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   X1                       10000 non-null  Int64         
 1   source_group             10000 non-null  str           
 2   ID_T01                   10000 non-null  object        
 3   Produktionsdatum         10000 non-null  datetime64[us]
 4   Herstellernummer         10000 non-null  object        
 5   Werksnummer              10000 non-null  object        
 6   Fehlerhaft               10000 non-null  Int8          
 7   Fehlerhaft_Datum         2505 non-null   datetime64[us]
 8   Fehlerhaft_Fahrleistung  10000 non-null  Float64       
dtypes: Float64(1), Int64(1), Int8(1), datetime64[us](2), object(3), str(1)
memory usage: 664.2+ KB


In [11]:
test_clean.head()
test_clean["Fehlerhaft"].value_counts(dropna=False)
test_clean.groupby("Fehlerhaft")[
    ["Fehlerhaft_Datum", "Fehlerhaft_Fahrleistung"]
].count()

,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
Fehlerhaft,,
0,0,7495
1,2505,2505


In [12]:
test_clean.loc[
    test_clean["Fehlerhaft"] == 0,
    "Fehlerhaft_Fahrleistung"
].value_counts(dropna=False)

Fehlerhaft_Fahrleistung
0.0    7495
Name: count, dtype: Int64

In [13]:
test_clean.loc[
    test_clean["Fehlerhaft"] == 1,
    "Fehlerhaft_Fahrleistung"
].describe()

count          2505.0
mean     11936.074651
std        614.549429
min           10597.0
25%           11551.0
50%           12029.0
75%           12363.0
max           13222.0
Name: Fehlerhaft_Fahrleistung, dtype: Float64

In [14]:
print(
    "Faulty without failure date:",
    (
        (test_clean["Fehlerhaft"] == 1)
        & test_clean["Fehlerhaft_Datum"].isna()
    ).sum()
)

print(
    "Non-faulty with failure date:",
    (
        (test_clean["Fehlerhaft"] == 0)
        & test_clean["Fehlerhaft_Datum"].notna()
    ).sum()
)

print(
    "Faulty with zero failure mileage:",
    (
        (test_clean["Fehlerhaft"] == 1)
        & (test_clean["Fehlerhaft_Fahrleistung"] == 0)
    ).sum()
)

Faulty without failure date: 0
Non-faulty with failure date: 0
Faulty with zero failure mileage: 0


In [15]:
from pathlib import Path
import pandas as pd

output = Path("data/Einzelteil/Einzelteil_T01.csv")

if output.exists():
    output.unlink()

total_rows = 0
first_chunk = True

source_counts = {
    "x": 0,
    "y": 0,
    "base": 0,
    "unknown": 0
}

for chunk in read_t01_chunks(rows_per_chunk=50_000):

    clean = clean_t01_chunk(chunk)
    clean = convert_t01_types(clean)

    counts = clean["source_group"].value_counts()

    for group in source_counts:
        source_counts[group] += counts.get(group, 0)

    faulty_without_date = (
        (clean["Fehlerhaft"] == 1)
        & clean["Fehlerhaft_Datum"].isna()
    ).sum()

    nonfaulty_with_date = (
        (clean["Fehlerhaft"] == 0)
        & clean["Fehlerhaft_Datum"].notna()
    ).sum()

    if faulty_without_date > 0:
        raise ValueError(
            f"Found {faulty_without_date} faulty rows without failure date"
        )

    if nonfaulty_with_date > 0:
        raise ValueError(
            f"Found {nonfaulty_with_date} non-faulty rows with failure date"
        )

    clean.to_csv(
        output,
        mode="a",
        header=first_chunk,
        index=False
    )

    first_chunk = False
    total_rows += len(clean)

    print(
        f"\rProcessed {total_rows:,} / 3,204,104 rows "
        f"({total_rows / 3_204_104:.1%})",
        end=""
    )

print()
print("Finished")
print(f"Total rows: {total_rows:,}")
print(f"Source groups: {source_counts}")

Processed 3,204,104 / 3,204,104 rows (100.0%)
Finished
Total rows: 3,204,104
Source groups: {'x': np.int64(1281642), 'y': np.int64(1281642), 'base': np.int64(640820), 'unknown': 0}


In [16]:
import pandas as pd

path = "data/Einzelteil/Einzelteil_T01.csv"

df = pd.read_csv(
    path,
    parse_dates=[
        "Produktionsdatum",
        "Fehlerhaft_Datum"
    ]
)

print("Shape:")
print(df.shape)

print("\nInfo:")
df.info()

print("\nMissing values:")
print(df.isna().sum())

print("\nFehlerhaft counts:")
print(df["Fehlerhaft"].value_counts(dropna=False))

print("\nSource groups:")
print(df["source_group"].value_counts(dropna=False))

print("\nIntegrity checks:")

faulty_without_date = (
    (df["Fehlerhaft"] == 1)
    & df["Fehlerhaft_Datum"].isna()
).sum()

nonfaulty_with_date = (
    (df["Fehlerhaft"] == 0)
    & df["Fehlerhaft_Datum"].notna()
).sum()

faulty_zero_mileage = (
    (df["Fehlerhaft"] == 1)
    & (df["Fehlerhaft_Fahrleistung"] == 0)
).sum()

print("Faulty without failure date:", faulty_without_date)
print("Non-faulty with failure date:", nonfaulty_with_date)
print("Faulty with zero failure mileage:", faulty_zero_mileage)

print("\nID_T01 checks:")
print("Rows:", len(df))
print("Unique ID_T01:", df["ID_T01"].nunique(dropna=False))
print("Duplicated ID_T01:", df["ID_T01"].duplicated().sum())
print("Missing ID_T01:", df["ID_T01"].isna().sum())

print("\nX1 checks:")
print("Unique X1:", df["X1"].nunique(dropna=False))
print("Duplicated X1:", df["X1"].duplicated().sum())
print("Missing X1:", df["X1"].isna().sum())

print("\nExpected source counts:")
expected = {
    "x": 1281642,
    "y": 1281642,
    "base": 640820
}

actual = df["source_group"].value_counts().to_dict()

for group, expected_count in expected.items():
    actual_count = actual.get(group, 0)
    print(
        f"{group}: {actual_count:,} "
        f"(expected {expected_count:,}) "
        f"{'OK' if actual_count == expected_count else 'MISMATCH'}"
    )

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

Shape:
(3204104, 9)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 3204104 entries, 0 to 3204103
Data columns (total 9 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   X1                       int64         
 1   source_group             str           
 2   ID_T01                   str           
 3   Produktionsdatum         datetime64[us]
 4   Herstellernummer         int64         
 5   Werksnummer              int64         
 6   Fehlerhaft               int64         
 7   Fehlerhaft_Datum         datetime64[us]
 8   Fehlerhaft_Fahrleistung  float64       
dtypes: datetime64[us](2), float64(1), int64(4), str(2)
memory usage: 220.0 MB

Missing values:
X1                               0
source_group                     0
ID_T01                           0
Produktionsdatum                 0
Herstellernummer                 0
Werksnummer                      0
Fehlerhaft                       0
Fehlerhaft_Datum           2499776
F

,X1,source_group,ID_T01,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,660,x,1-201-2011-247,2008-11-07,201,2011,0,NaT,0.0
1,661,x,1-201-2011-429,2008-11-07,201,2011,0,NaT,0.0
2,662,x,1-201-2011-363,2008-11-07,201,2011,1,2009-09-30,12983.0
3,663,x,1-201-2011-30,2008-11-07,201,2011,0,NaT,0.0
4,664,x,1-201-2011-72,2008-11-07,201,2011,1,2009-09-30,12983.0



Last 5 rows:


,X1,source_group,ID_T01,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
3204099,3198824,base,1-204-2044-638601,2016-10-29,204,2044,0,NaT,0.00
3204100,3203033,base,1-204-2044-640072,2016-11-03,204,2044,0,NaT,0.00
3204101,3203037,base,1-204-2044-638951,2016-11-03,204,2044,1,2018-06-04,35509.67
3204102,3203719,base,1-204-2044-638770,2016-11-04,204,2044,1,2018-06-05,123579.00
3204103,3203725,base,1-204-2044-639891,2016-11-04,204,2044,0,NaT,0.00


In [17]:
from pathlib import Path

folder = Path("data/Einzelteil")

txt_files = sorted(folder.glob("Einzelteil_T*.txt"))

print(f"Found {len(txt_files)} TXT files\n")

for path in txt_files:
    size_mb = path.stat().st_size / (1024 ** 2)

    with path.open("rb") as f:
        sample = f.read(64 * 1024)

    newline_count = sample.count(b"\n")
    weird_delimiter = b" | | " in sample

    print(
        f"{path.name:<22} "
        f"{size_mb:>8.1f} MB   "
        f"newlines(first 64KB)={newline_count:<4}   "
        f"' | | '={weird_delimiter}"
    )

Found 16 TXT files

Einzelteil_T01.txt        609.2 MB   newlines(first 64KB)=0      ' | | '=True
Einzelteil_T02.txt        320.6 MB   newlines(first 64KB)=0      ' | | '=False
Einzelteil_T03.txt         87.7 MB   newlines(first 64KB)=0      ' | | '=False
Einzelteil_T07.txt         29.4 MB   newlines(first 64KB)=0      ' | | '=False
Einzelteil_T09.txt         34.1 MB   newlines(first 64KB)=0      ' | | '=False
Einzelteil_T11.txt        180.4 MB   newlines(first 64KB)=0      ' | | '=False
Einzelteil_T16.txt        154.1 MB   newlines(first 64KB)=0      ' | | '=True
Einzelteil_T20.txt         17.5 MB   newlines(first 64KB)=0      ' | | '=True
Einzelteil_T22.txt        269.9 MB   newlines(first 64KB)=0      ' | | '=False
Einzelteil_T24.txt         80.2 MB   newlines(first 64KB)=0      ' | | '=False
Einzelteil_T27.txt         17.5 MB   newlines(first 64KB)=0      ' | | '=True
Einzelteil_T31.txt        201.1 MB   newlines(first 64KB)=0      ' | | '=False
Einzelteil_T34.txt         87.9 MB  

In [18]:
from pathlib import Path

folder = Path("data/Einzelteil")

known_weird = {
    "Einzelteil_T01.txt",
    "Einzelteil_T16.txt",
    "Einzelteil_T20.txt",
    "Einzelteil_T27.txt",
    "Einzelteil_T34.txt"
}

for path in sorted(folder.glob("Einzelteil_T*.txt")):
    if path.name in known_weird:
        continue

    with path.open("rb") as f:
        sample = f.read(2000).decode("utf-8", errors="replace")

    print("=" * 100)
    print(path.name)
    print("=" * 100)
    print(repr(sample[:1000]))
    print()

Einzelteil_T02.txt
'"X1"  "ID_T02.x"  "Produktionsdatum.x"  "Herstellernummer.x"  "Werksnummer.x"  "Fehlerhaft.x"  "Fehlerhaft_Datum.x"  "Fehlerhaft_Fahrleistung.x"  "ID_T02.y"  "Produktionsdatum.y"  "Herstellernummer.y"  "Werksnummer.y"  "Fehlerhaft.y"  "Fehlerhaft_Datum.y"  "Fehlerhaft_Fahrleistung.y"\t"1"  4  "2-201-2011-239"  2008-11-07  "201"  2011  1  2010-04-09  38354.1589041096  NA  NA  NA  NA  NA  NA  NA\t"2"  5  "2-201-2011-304"  2008-11-07  "201"  2011  0  NA  0  NA  NA  NA  NA  NA  NA  NA\t"3"  9  "2-201-2011-125"  2008-11-07  "201"  2011  1  2010-04-09  38354.1589041096  NA  NA  NA  NA  NA  NA  NA\t"4"  17  "2-201-2011-55"  2008-11-07  "201"  2011  0  NA  0  NA  NA  NA  NA  NA  NA  NA\t"5"  19  "2-201-2011-133"  2008-11-07  "201"  2011  0  NA  0  NA  NA  NA  NA  NA  NA  NA\t"6"  22  "2-201-2011-121"  2008-11-07  "201"  2011  0  NA  0  NA  NA  NA  NA  NA  NA  NA\t"7"  23  "2-201-2011-245"  2008-11-07  "201"  2011  0  NA  0  NA  NA  NA  NA  NA  NA  NA\t"8"  27  "2-201-2011-1

In [19]:
import re
import pandas as pd
from pathlib import Path

FOLDER = Path("data/Einzelteil")

FORMATS = {
    "T01": {"field": rb" \| \| ", "row": None},
    "T02": {"field": rb" {2,}",   "row": b"\t"},
    "T03": {"field": rb"\|",      "row": b"\x0b"},
    "T07": {"field": rb"\t",      "row": None},
    "T09": {"field": rb"\\",      "row": b"\x0b"},
    "T11": {"field": rb"\t",      "row": b"\x0c"},
    "T16": {"field": rb" \| \| ", "row": None},
    "T20": {"field": rb" \| \| ", "row": None},
    "T22": {"field": rb"\t",      "row": None},
    "T24": {"field": rb" {2,}",   "row": b"\x0c"},
    "T27": {"field": rb" \| \| ", "row": None},
    "T31": {"field": rb" {2,}",   "row": b"\x08"},
    "T34": {"field": rb" \| \| ", "row": None},
    "T35": {"field": rb"\\",      "row": None},
    "T36": {"field": rb" {2,}",   "row": None},
    "T39": {"field": rb"\\",      "row": b"\x07"},
}


def decode_value(value):
    value = value.strip().decode("utf-8", errors="replace")

    if len(value) >= 2 and value.startswith('"') and value.endswith('"'):
        value = value[1:-1]

    if value == "NA":
        return pd.NA

    return value


def iter_records(path, row_sep, read_size=4 * 1024 * 1024):
    carry = b""

    with open(path, "rb") as f:
        while True:
            chunk = f.read(read_size)

            if not chunk:
                break

            parts = (carry + chunk).split(row_sep)
            carry = parts.pop()

            for record in parts:
                if record.strip():
                    yield record

        if carry.strip():
            yield carry


def split_fields(record, pattern):
    return [
        part
        for part in re.split(pattern, record.strip())
        if part.strip()
    ]


def iter_tokens(path, pattern, read_size=4 * 1024 * 1024):
    splitter = re.compile(pattern)
    carry = b""

    with open(path, "rb") as f:
        while True:
            chunk = f.read(read_size)

            if not chunk:
                break

            data = carry + chunk
            last_end = 0
            found = False

            for match in splitter.finditer(data):
                found = True
                yield data[last_end:match.start()]
                last_end = match.end()

            if found:
                carry = data[last_end:]
            else:
                carry = data

            if len(carry) > 32 * 1024 * 1024:
                raise RuntimeError(
                    f"Parser buffer exceeded 32 MB for {path.name}"
                )

        if carry.strip():
            yield carry


def read_with_row_separator(
    path,
    field_pattern,
    row_sep,
    rows_per_chunk
):
    records = iter_records(path, row_sep)

    header_record = next(records)
    columns = [
        decode_value(x)
        for x in split_fields(header_record, field_pattern)
    ]

    rows = []
    expected_row = 1

    for record in records:
        fields = split_fields(record, field_pattern)

        row_number = decode_value(fields[0])

        if str(row_number) != str(expected_row):
            raise ValueError(
                f"{path.name}: expected row {expected_row}, "
                f"found {row_number}"
            )

        values = [
            decode_value(x)
            for x in fields[1:]
        ]

        if len(values) != len(columns):
            raise ValueError(
                f"{path.name}, row {expected_row}: "
                f"{len(values)} values for {len(columns)} columns"
            )

        rows.append(values)
        expected_row += 1

        if len(rows) == rows_per_chunk:
            yield pd.DataFrame(rows, columns=columns)
            rows = []

    if rows:
        yield pd.DataFrame(rows, columns=columns)


def read_without_row_separator(
    path,
    field_pattern,
    rows_per_chunk
):
    tokens = iter_tokens(path, field_pattern)

    boundary = re.compile(rb'^(.*?)\s*"(\d+)"$')

    header = []
    columns = None
    current_row = []
    rows = []
    expected_row = 1

    for token in tokens:
        token = token.strip()

        match = boundary.match(token)

        if match:
            prefix = match.group(1).strip()
            number = int(match.group(2))
        else:
            prefix = b""
            number = -1

        is_boundary = bool(prefix) and number == expected_row

        if not is_boundary:
            if columns is None:
                header.append(token)
            else:
                current_row.append(token)

            continue

        if columns is None:
            header.append(prefix)

            columns = [
                decode_value(x)
                for x in header
            ]

        else:
            current_row.append(prefix)

            values = [
                decode_value(x)
                for x in current_row
            ]

            if len(values) != len(columns):
                raise ValueError(
                    f"{path.name}, row {expected_row - 1}: "
                    f"{len(values)} values for {len(columns)} columns"
                )

            rows.append(values)
            current_row = []

            if len(rows) == rows_per_chunk:
                yield pd.DataFrame(rows, columns=columns)
                rows = []

        expected_row += 1

    if current_row:
        values = [
            decode_value(x)
            for x in current_row
        ]

        if len(values) != len(columns):
            raise ValueError(
                f"{path.name}, final row: "
                f"{len(values)} values for {len(columns)} columns"
            )

        rows.append(values)

    if rows:
        yield pd.DataFrame(rows, columns=columns)


def read_broken_einzelteil(part, rows_per_chunk=50_000):
    config = FORMATS[part]

    path = FOLDER / f"Einzelteil_{part}.txt"

    if config["row"] is None:
        yield from read_without_row_separator(
            path,
            config["field"],
            rows_per_chunk
        )
    else:
        yield from read_with_row_separator(
            path,
            config["field"],
            config["row"],
            rows_per_chunk
        )

In [20]:
for part in FORMATS:
    try:
        test = next(
            read_broken_einzelteil(
                part,
                rows_per_chunk=5
            )
        )

        print(
            f"{part}: OK | "
            f"{len(test.columns)} columns | "
            f"{test.shape[0]} test rows"
        )

    except Exception as e:
        print(
            f"{part}: FAILED | "
            f"{type(e).__name__}: {e}"
        )

T01: OK | 22 columns | 5 test rows
T02: OK | 15 columns | 5 test rows
T03: OK | 9 columns | 5 test rows
T07: OK | 9 columns | 5 test rows
T09: OK | 15 columns | 5 test rows
T11: OK | 9 columns | 5 test rows
T16: OK | 22 columns | 5 test rows
T20: OK | 9 columns | 5 test rows
T22: OK | 22 columns | 5 test rows
T24: OK | 22 columns | 5 test rows
T27: OK | 9 columns | 5 test rows
T31: OK | 9 columns | 5 test rows
T34: OK | 9 columns | 5 test rows
T35: OK | 15 columns | 5 test rows
T36: OK | 9 columns | 5 test rows
T39: OK | 15 columns | 5 test rows


In [21]:
import pandas as pd
from pathlib import Path

In [22]:
def normalize_einzelteil_chunk(chunk, part):
    id_col = f"ID_{part}"

    out = pd.DataFrame(index=chunk.index)

    out["X1"] = pd.to_numeric(
        chunk["X1"],
        errors="coerce"
    ).astype("Int64")

    has_suffix_groups = f"{id_col}.x" in chunk.columns

    if has_suffix_groups:
        source = pd.Series(
            pd.NA,
            index=chunk.index,
            dtype="string"
        )

        if f"{id_col}.x" in chunk.columns:
            mask = chunk[f"{id_col}.x"].notna()
            source.loc[mask] = "x"

        if f"{id_col}.y" in chunk.columns:
            mask = (
                source.isna()
                & chunk[f"{id_col}.y"].notna()
            )
            source.loc[mask] = "y"

        if id_col in chunk.columns:
            mask = (
                source.isna()
                & chunk[id_col].notna()
            )
            source.loc[mask] = "base"

        if source.isna().any():
            raise ValueError(
                f"{part}: "
                f"{source.isna().sum()} rows have no source group"
            )

        out["source_group"] = source

        fields = [
            id_col,
            "Produktionsdatum",
            "Herstellernummer",
            "Werksnummer",
            "Fehlerhaft",
            "Fehlerhaft_Datum",
            "Fehlerhaft_Fahrleistung"
        ]

        for field in fields:
            candidates = [
                col
                for col in [
                    f"{field}.x",
                    f"{field}.y",
                    field
                ]
                if col in chunk.columns
            ]

            if not candidates:
                raise KeyError(
                    f"{part}: no column found for '{field}'"
                )

            combined = chunk[candidates[0]]

            for candidate in candidates[1:]:
                combined = combined.combine_first(
                    chunk[candidate]
                )

            out[field] = combined

    else:
        out["source_group"] = "base"

        required = [
            id_col,
            "Herstellernummer",
            "Werksnummer",
            "Fehlerhaft",
            "Fehlerhaft_Datum",
            "Fehlerhaft_Fahrleistung"
        ]

        missing = [
            col
            for col in required
            if col not in chunk.columns
        ]

        if missing:
            raise KeyError(
                f"{part}: missing columns {missing}"
            )

        out[id_col] = chunk[id_col]
        out["Herstellernummer"] = chunk["Herstellernummer"]
        out["Werksnummer"] = chunk["Werksnummer"]
        out["Fehlerhaft"] = chunk["Fehlerhaft"]
        out["Fehlerhaft_Datum"] = chunk["Fehlerhaft_Datum"]
        out["Fehlerhaft_Fahrleistung"] = (
            chunk["Fehlerhaft_Fahrleistung"]
        )

        if "Produktionsdatum" in chunk.columns:
            out["Produktionsdatum"] = chunk["Produktionsdatum"]

        elif (
            "Produktionsdatum_Origin_01011970" in chunk.columns
            and "origin" in chunk.columns
        ):
            days = pd.to_numeric(
                chunk["Produktionsdatum_Origin_01011970"],
                errors="coerce"
            )

            origin = pd.to_datetime(
                chunk["origin"],
                format="%d-%m-%Y",
                errors="coerce"
            )

            out["Produktionsdatum"] = (
                origin
                + pd.to_timedelta(days, unit="D")
            )

        else:
            raise KeyError(
                f"{part}: cannot determine Produktionsdatum. "
                f"Available columns: {chunk.columns.tolist()}"
            )

    out = out.rename(
        columns={id_col: "ID"}
    )

    out["ID"] = out["ID"].astype("string")

    out["Herstellernummer"] = (
        out["Herstellernummer"]
        .astype("string")
    )

    out["Werksnummer"] = (
        out["Werksnummer"]
        .astype("string")
    )

    out["Produktionsdatum"] = pd.to_datetime(
        out["Produktionsdatum"],
        errors="coerce"
    )

    out["Fehlerhaft_Datum"] = pd.to_datetime(
        out["Fehlerhaft_Datum"],
        errors="coerce"
    )

    out["Fehlerhaft"] = pd.to_numeric(
        out["Fehlerhaft"],
        errors="coerce"
    ).astype("Int8")

    out["Fehlerhaft_Fahrleistung"] = pd.to_numeric(
        out["Fehlerhaft_Fahrleistung"],
        errors="coerce"
    ).astype("Float64")

    return out[
        [
            "X1",
            "source_group",
            "ID",
            "Produktionsdatum",
            "Herstellernummer",
            "Werksnummer",
            "Fehlerhaft",
            "Fehlerhaft_Datum",
            "Fehlerhaft_Fahrleistung"
        ]
    ]

In [23]:
for part in FORMATS:
    try:
        raw = next(
            read_broken_einzelteil(
                part,
                rows_per_chunk=5
            )
        )

        print(f"\n{'=' * 80}")
        print(part)
        print(f"{'=' * 80}")

        for i, col in enumerate(raw.columns, start=1):
            print(f"{i:2d}. {repr(col)}")

    except Exception as e:
        print(
            f"\n{part}: FAILED | "
            f"{type(e).__name__}: {e}"
        )


T01
 1. 'X1'
 2. 'ID_T01.x'
 3. 'Produktionsdatum.x'
 4. 'Herstellernummer.x'
 5. 'Werksnummer.x'
 6. 'Fehlerhaft.x'
 7. 'Fehlerhaft_Datum.x'
 8. 'Fehlerhaft_Fahrleistung.x'
 9. 'ID_T01.y'
10. 'Produktionsdatum.y'
11. 'Herstellernummer.y'
12. 'Werksnummer.y'
13. 'Fehlerhaft.y'
14. 'Fehlerhaft_Datum.y'
15. 'Fehlerhaft_Fahrleistung.y'
16. 'ID_T01'
17. 'Produktionsdatum'
18. 'Herstellernummer'
19. 'Werksnummer'
20. 'Fehlerhaft'
21. 'Fehlerhaft_Datum'
22. 'Fehlerhaft_Fahrleistung'

T02
 1. 'X1'
 2. 'ID_T02.x'
 3. 'Produktionsdatum.x'
 4. 'Herstellernummer.x'
 5. 'Werksnummer.x'
 6. 'Fehlerhaft.x'
 7. 'Fehlerhaft_Datum.x'
 8. 'Fehlerhaft_Fahrleistung.x'
 9. 'ID_T02.y'
10. 'Produktionsdatum.y'
11. 'Herstellernummer.y'
12. 'Werksnummer.y'
13. 'Fehlerhaft.y'
14. 'Fehlerhaft_Datum.y'
15. 'Fehlerhaft_Fahrleistung.y'

T03
 1. 'X1'
 2. 'ID_T03'
 3. 'Herstellernummer'
 4. 'Werksnummer'
 5. 'Fehlerhaft'
 6. 'Fehlerhaft_Datum'
 7. 'Fehlerhaft_Fahrleistung'
 8. 'Produktionsdatum_Origin_01011970'
 9. 

In [24]:
FORMATS["T27"]["row"] = b"\x07"
raw = next(
    read_broken_einzelteil(
        "T27",
        rows_per_chunk=5
    )
)

print(raw.columns.tolist())

['X1', 'ID_T27', 'Herstellernummer', 'Werksnummer', 'Fehlerhaft', 'Fehlerhaft_Datum', 'Fehlerhaft_Fahrleistung', 'Produktionsdatum_Origin_01011970', 'origin']


In [25]:
from pathlib import Path
import pandas as pd

FOLDER = Path("data/Einzelteil")

FORMATS["T27"]["row"] = b"\x07"

parts_to_process = list(FORMATS.keys())

summaries = {}

for part in parts_to_process:
    output = FOLDER / f"Einzelteil_{part}.csv"

    if output.exists():
        output.unlink()

    total_rows = 0
    first_chunk = True

    source_counts = {
        "x": 0,
        "y": 0,
        "base": 0
    }

    overlap_rows = 0
    missing_source_rows = 0
    missing_id = 0
    faulty_without_date = 0
    nonfaulty_with_date = 0

    for raw in read_broken_einzelteil(
        part,
        rows_per_chunk=50_000
    ):
        id_col = f"ID_{part}"

        if f"{id_col}.x" in raw.columns:
            source_columns = [
                col
                for col in [
                    f"{id_col}.x",
                    f"{id_col}.y",
                    id_col
                ]
                if col in raw.columns
            ]

            populated = (
                raw[source_columns]
                .notna()
                .sum(axis=1)
            )

            overlaps = int(
                (populated > 1).sum()
            )

            missing_sources = int(
                (populated == 0).sum()
            )

            if overlaps > 0:
                raise ValueError(
                    f"{part}: {overlaps} rows have "
                    f"multiple source groups"
                )

            if missing_sources > 0:
                raise ValueError(
                    f"{part}: {missing_sources} rows have "
                    f"no source group"
                )

            overlap_rows += overlaps
            missing_source_rows += missing_sources

        clean = normalize_einzelteil_chunk(
            raw,
            part
        )

        counts = (
            clean["source_group"]
            .value_counts()
        )

        for group in source_counts:
            source_counts[group] += int(
                counts.get(group, 0)
            )

        missing_id += int(
            clean["ID"].isna().sum()
        )

        faulty_without_date += int(
            (
                (clean["Fehlerhaft"] == 1)
                & clean["Fehlerhaft_Datum"].isna()
            ).sum()
        )

        nonfaulty_with_date += int(
            (
                (clean["Fehlerhaft"] == 0)
                & clean["Fehlerhaft_Datum"].notna()
            ).sum()
        )

        clean.to_csv(
            output,
            mode="a",
            header=first_chunk,
            index=False
        )

        first_chunk = False
        total_rows += len(clean)

        print(
            f"\r{part}: {total_rows:,} rows processed",
            end=""
        )

    summaries[part] = {
        "rows": total_rows,
        "x": source_counts["x"],
        "y": source_counts["y"],
        "base": source_counts["base"],
        "overlap_rows": overlap_rows,
        "missing_source_rows": missing_source_rows,
        "missing_ID": missing_id,
        "faulty_without_date": faulty_without_date,
        "nonfaulty_with_date": nonfaulty_with_date
    }

    print(
        f"\r{part}: finished — {total_rows:,} rows"
        + " " * 20
    )

summary_df = pd.DataFrame.from_dict(
    summaries,
    orient="index"
)

summary_df.index.name = "part"

display(summary_df)

T01: finished — 3,204,104 rows                    
T02: finished — 3,204,104 rows                    
T03: finished — 1,192,630 rows                    
T07: finished — 409,422 rows                    
T09: finished — 409,422 rows                    
T11: finished — 2,385,260 rows                    
T16: finished — 818,844 rows                    
T20: finished — 163,769 rows                    
T22: finished — 2,563,283 rows                    
T24: finished — 640,821 rows                    
T27: finished — 163,769 rows                    
T31: finished — 2,385,260 rows                    
T34: finished — 818,844 rows                    
T35: finished — 818,844 rows                    
T36: finished — 512,354 rows                    
T39: finished — 306,490 rows                    


,rows,x,y,base,overlap_rows,missing_source_rows,missing_ID,faulty_without_date,nonfaulty_with_date
part,,,,,,,,,
T01,3204104,1281642,1281642,640820,0,0,0,0,0
T02,3204104,961231,2242873,0,0,0,0,0,0
T03,1192630,0,0,1192630,0,0,0,0,0
T07,409422,0,0,409422,0,0,0,0,0
T09,409422,122826,286596,0,0,0,0,0,0
T11,2385260,0,0,2385260,0,0,0,0,0
T16,818844,245653,163769,409422,0,0,0,0,0
T20,163769,0,0,163769,0,0,0,0,0
T22,2563283,1281642,1025313,256328,0,0,0,0,0


In [26]:
from pathlib import Path
import pandas as pd

FOLDER = Path("data/Einzelteil")

expected_columns = [
    "X1",
    "source_group",
    "ID",
    "Produktionsdatum",
    "Herstellernummer",
    "Werksnummer",
    "Fehlerhaft",
    "Fehlerhaft_Datum",
    "Fehlerhaft_Fahrleistung"
]

verification = []

for part in FORMATS:
    path = FOLDER / f"Einzelteil_{part}.csv"

    if not path.exists():
        verification.append({
            "part": part,
            "file_exists": False
        })
        continue

    rows = 0
    missing_id = 0
    faulty_without_date = 0
    nonfaulty_with_date = 0
    wrong_id_prefix = 0
    duplicate_id = 0
    columns_ok = None

    seen_ids = set()

    for chunk in pd.read_csv(
        path,
        chunksize=100_000,
        parse_dates=[
            "Produktionsdatum",
            "Fehlerhaft_Datum"
        ]
    ):
        if columns_ok is None:
            columns_ok = (
                chunk.columns.tolist()
                == expected_columns
            )

        rows += len(chunk)

        missing_id += int(
            chunk["ID"].isna().sum()
        )

        faulty_without_date += int(
            (
                (chunk["Fehlerhaft"] == 1)
                & chunk["Fehlerhaft_Datum"].isna()
            ).sum()
        )

        nonfaulty_with_date += int(
            (
                (chunk["Fehlerhaft"] == 0)
                & chunk["Fehlerhaft_Datum"].notna()
            ).sum()
        )

        part_number = part[1:].lstrip("0")

        wrong_id_prefix += int(
            (
                ~chunk["ID"]
                .astype(str)
                .str.startswith(part_number + "-")
            ).sum()
        )

        ids = chunk["ID"].astype(str)

        duplicate_id += int(
            ids.duplicated().sum()
        )

        duplicate_id += sum(
            id_ in seen_ids
            for id_ in ids.unique()
        )

        seen_ids.update(ids.unique())

    expected_rows = int(
        summary_df.loc[part, "rows"]
    )

    verification.append({
        "part": part,
        "file_exists": True,
        "columns_ok": columns_ok,
        "rows": rows,
        "expected_rows": expected_rows,
        "rows_ok": rows == expected_rows,
        "missing_ID": missing_id,
        "wrong_ID_prefix": wrong_id_prefix,
        "duplicate_ID": duplicate_id,
        "faulty_without_date": faulty_without_date,
        "nonfaulty_with_date": nonfaulty_with_date
    })

verification_df = (
    pd.DataFrame(verification)
    .set_index("part")
)

display(verification_df)

print(
    "\nTotal cleaned rows:",
    f"{verification_df['rows'].sum():,}"
)

,file_exists,columns_ok,rows,expected_rows,rows_ok,missing_ID,wrong_ID_prefix,duplicate_ID,faulty_without_date,nonfaulty_with_date
part,,,,,,,,,,
T01,True,True,3204104,3204104,True,0,0,0,0,0
T02,True,True,3204104,3204104,True,0,0,0,0,0
T03,True,True,1192630,1192630,True,0,0,0,0,0
T07,True,True,409422,409422,True,0,0,0,0,0
T09,True,True,409422,409422,True,0,0,0,0,0
T11,True,True,2385260,2385260,True,0,0,0,0,0
T16,True,True,818844,818844,True,0,0,0,0,0
T20,True,True,163769,163769,True,0,0,0,0,0
T22,True,True,2563283,2563283,True,0,0,0,0,0



Total cleaned rows: 19,997,220


Components Data cleanup

In [29]:
import re
import pandas as pd
from pathlib import Path

KOMPONENTE_FOLDER = Path("data/Komponente")

def detect_component_format(path):
    with path.open("rb") as f:
        sample = f.read(64 * 1024)

    head = sample[:4096]

    if b" | | " in head:
        field_pattern = rb" \| \| "
        field_name = "| |"

    elif head.count(b"II") >= 5:
        field_pattern = rb"II"
        field_name = "II"

    elif b"\\" in head:
        field_pattern = rb"\\"
        field_name = "\\"

    elif b"|" in head:
        field_pattern = rb"\|"
        field_name = "|"

    elif re.search(rb'" {2,}"', head):
        field_pattern = rb" {2,}"
        field_name = "spaces"

    elif b"\t" in head:
        field_pattern = rb"\t"
        field_name = "TAB"

    else:
        raise ValueError(
            f"{path.name}: unknown field separator"
        )

    row_sep = None
    row_name = "glued"

    control_separators = [
        (b"\x07", "\\x07"),
        (b"\x08", "\\x08"),
        (b"\x0b", "\\x0b"),
        (b"\x0c", "\\x0c"),
        (b"\r\n", "CRLF"),
        (b"\n", "LF"),
    ]

    for sep, name in control_separators:
        if sep in sample:
            row_sep = sep
            row_name = name
            break

    if (
        row_sep is None
        and field_name != "TAB"
        and b"\t" in sample
    ):
        row_sep = b"\t"
        row_name = "TAB"

    return {
        "field": field_pattern,
        "field_name": field_name,
        "row": row_sep,
        "row_name": row_name
    }

In [33]:
def read_component_txt(path, rows_per_chunk=50_000):
    config = detect_component_format(path)

    if config["row"] is None:
        yield from read_without_row_separator(
            path,
            config["field"],
            rows_per_chunk
        )
    else:
        yield from read_with_row_separator(
            path,
            config["field"],
            config["row"],
            rows_per_chunk
        )

In [34]:
def component_id_column(family):
    if family.startswith("K1"):
        return "ID_Motor"

    if family.startswith("K2"):
        return "ID_Sitze"

    if family.startswith("K3"):
        return "ID_Schaltung"

    if family in ["K4", "K6"]:
        return "ID_Karosserie"

    raise ValueError(
        f"Unknown component family: {family}"
    )


def normalize_component_chunk(df, family):

    id_col = component_id_column(family)

    fields = [
        id_col,
        "Produktionsdatum",
        "Herstellernummer",
        "Werksnummer",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung",
    ]

    clean = pd.DataFrame(index=df.index)

    # x/y or x/y/base structure
    if f"{id_col}.x" in df.columns:

        for field in fields:

            candidates = [
                col
                for col in [
                    f"{field}.x",
                    f"{field}.y",
                    field
                ]
                if col in df.columns
            ]

            if not candidates:
                continue

            combined = df[candidates[0]]

            for col in candidates[1:]:
                combined = combined.combine_first(
                    df[col]
                )

            clean[field] = combined

    # normal structure
    else:

        for field in fields:
            if field in df.columns:
                clean[field] = df[field]

        # date stored as days since origin
        if (
            "Produktionsdatum" not in df.columns
            and
            "Produktionsdatum_Origin_01011970"
            in df.columns
        ):
            days = pd.to_numeric(
                df[
                    "Produktionsdatum_Origin_01011970"
                ],
                errors="coerce"
            )

            origin = pd.to_datetime(
                df["origin"],
                format="%d-%m-%Y",
                errors="coerce"
            )

            clean["Produktionsdatum"] = (
                origin
                + pd.to_timedelta(days, unit="D")
            )

    clean = clean.rename(
        columns={id_col: "ID"}
    )

    clean.insert(
        0,
        "Komponententyp",
        family
    )

    clean["ID"] = clean["ID"].astype("string")

    clean["Produktionsdatum"] = pd.to_datetime(
        clean["Produktionsdatum"],
        errors="coerce"
    )

    clean["Herstellernummer"] = (
        pd.to_numeric(
            clean["Herstellernummer"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    clean["Werksnummer"] = (
        pd.to_numeric(
            clean["Werksnummer"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    clean["Fehlerhaft"] = pd.to_numeric(
        clean["Fehlerhaft"],
        errors="coerce"
    ).astype("Int8")

    clean["Fehlerhaft_Datum"] = pd.to_datetime(
        clean["Fehlerhaft_Datum"],
        errors="coerce"
    )

    clean["Fehlerhaft_Fahrleistung"] = (
        pd.to_numeric(
            clean["Fehlerhaft_Fahrleistung"],
            errors="coerce"
        )
        .astype("Float64")
    )

    return clean[
        [
            "Komponententyp",
            "ID",
            "Produktionsdatum",
            "Herstellernummer",
            "Werksnummer",
            "Fehlerhaft",
            "Fehlerhaft_Datum",
            "Fehlerhaft_Fahrleistung",
        ]
    ]

In [35]:
relevant_families = {
    "K4",
    "K3SG1",
    "K3AG1",
    "K2ST1",
    "K2LE1",
    "K1DI1",
    "K1BE1",
    "K6",
    "K3SG2",
    "K3AG2",
    "K2ST2",
    "K2LE2",
    "K1BE2",
    "K1DI2",
}

txt_files = sorted(
    path
    for path in KOMPONENTE_FOLDER.glob(
        "Komponente_*.txt"
    )
    if path.stem.replace(
        "Komponente_", ""
    ) in relevant_families
)

for path in txt_files:
    print(path.name)

Komponente_K1DI2.txt
Komponente_K2LE1.txt
Komponente_K2LE2.txt
Komponente_K2ST1.txt
Komponente_K3AG2.txt


In [36]:
for path in txt_files:

    print("=" * 100)
    print(path.name)

    family = path.stem.replace(
        "Komponente_", ""
    )

    try:
        config = detect_component_format(path)

        print(
            f"Detected: "
            f"field={config['field_name']}, "
            f"row={config['row_name']}"
        )

        raw = next(
            read_component_txt(
                path,
                rows_per_chunk=5
            )
        )

        print("Raw columns:")
        print(raw.columns.tolist())

        clean = normalize_component_chunk(
            raw,
            family
        )

        print(
            f"RESULT: OK | "
            f"{len(raw.columns)} -> "
            f"{len(clean.columns)} columns"
        )

        display(clean.head(2))

    except Exception as e:
        print(
            f"RESULT: FAILED | "
            f"{type(e).__name__}: {e}"
        )

    print()

Komponente_K1DI2.txt
Detected: field=\, row=TAB
Raw columns:
['X1', 'ID_Motor', 'Herstellernummer', 'Werksnummer', 'Fehlerhaft', 'Fehlerhaft_Datum', 'Fehlerhaft_Fahrleistung', 'Produktionsdatum_Origin_01011970', 'origin']
RESULT: OK | 9 -> 8 columns


,Komponententyp,ID,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,K1DI2,K1DI2-103-1031-2,2008-11-12,103,1031,0,NaT,0.0
1,K1DI2,K1DI2-103-1031-9,2008-11-13,103,1031,0,NaT,0.0



Komponente_K2LE1.txt
Detected: field=II, row=\x0b
Raw columns:
['X1', 'ID_Sitze.x', 'Produktionsdatum.x', 'Herstellernummer.x', 'Werksnummer.x', 'Fehlerhaft.x', 'Fehlerhaft_Datum.x', 'Fehlerhaft_Fahrleistung.x', 'ID_Sitze.y', 'Produktionsdatum.y', 'Herstellernummer.y', 'Werksnummer.y', 'Fehlerhaft.y', 'Fehlerhaft_Datum.y', 'Fehlerhaft_Fahrleistung.y']
RESULT: OK | 15 -> 8 columns


,Komponententyp,ID,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,K2LE1,K2LE1-109-1091-2,2008-11-12,109,1091,1,2010-10-18,37080.0
1,K2LE1,K2LE1-109-1091-1,2008-11-12,109,1091,0,NaT,0.0



Komponente_K2LE2.txt
Detected: field=\, row=glued
Raw columns:
['X1', 'ID_Sitze', 'Herstellernummer', 'Werksnummer', 'Fehlerhaft', 'Fehlerhaft_Datum', 'Fehlerhaft_Fahrleistung', 'Produktionsdatum_Origin_01011970', 'origin']
RESULT: OK | 9 -> 8 columns


,Komponententyp,ID,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,K2LE2,K2LE2-111-1111-1,2008-11-12,111,1111,0,NaT,0.0
1,K2LE2,K2LE2-111-1111-4,2008-11-13,111,1111,0,NaT,0.0



Komponente_K2ST1.txt
Detected: field=|, row=glued
Raw columns:
['X1', 'ID_Sitze', 'Produktionsdatum', 'Herstellernummer', 'Werksnummer', 'Fehlerhaft', 'Fehlerhaft_Datum', 'Fehlerhaft_Fahrleistung']
RESULT: OK | 8 -> 8 columns


,Komponententyp,ID,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,K2ST1,K2ST1-109-1092-5,2008-11-12,109,1092,1,2010-09-04,47634.0
1,K2ST1,K2ST1-109-1092-1,2008-11-12,109,1092,0,NaT,0.0



Komponente_K3AG2.txt
Detected: field=\, row=glued
Raw columns:
['X1', 'ID_Schaltung', 'Herstellernummer', 'Werksnummer', 'Fehlerhaft', 'Fehlerhaft_Datum', 'Fehlerhaft_Fahrleistung', 'Produktionsdatum_Origin_01011970', 'origin']
RESULT: OK | 9 -> 8 columns


,Komponententyp,ID,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,K3AG2,K3AG2-108-1082-9,2008-11-13,108,1082,0,NaT,0.0
1,K3AG2,K3AG2-108-1082-11,2008-11-13,108,1082,1,2010-03-02,31859.0


In [39]:
from pathlib import Path
import pandas as pd

KOMPONENTE_FOLDER = Path("data/Komponente")

relevant_families = {
    "K4",
    "K3SG1",
    "K3AG1",
    "K2ST1",
    "K2LE1",
    "K1DI1",
    "K1BE1",
    "K6",
    "K3SG2",
    "K3AG2",
    "K2ST2",
    "K2LE2",
    "K1BE2",
    "K1DI2",
}

canonical_columns = [
    "Komponententyp",
    "ID",
    "Produktionsdatum",
    "Herstellernummer",
    "Werksnummer",
    "Fehlerhaft",
    "Fehlerhaft_Datum",
    "Fehlerhaft_Fahrleistung",
]


def detect_separator(path):
    with open(
        path,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:
        first_line = f.readline()

    candidates = [",", ";", "\t", "|"]

    return max(
        candidates,
        key=lambda sep: first_line.count(sep)
    )


def component_id_column(family):
    if family.startswith("K1"):
        return "ID_Motor"

    if family.startswith("K2"):
        return "ID_Sitze"

    if family.startswith("K3"):
        return "ID_Schaltung"

    if family in {"K4", "K6"}:
        return "ID_Karosserie"

    raise ValueError(
        f"Unknown component family: {family}"
    )


def normalize_component_chunk(df, family):
    id_col = component_id_column(family)

    fields = [
        id_col,
        "Produktionsdatum",
        "Herstellernummer",
        "Werksnummer",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung",
    ]

    if "ID" in df.columns:
        required = [
            "ID",
            "Produktionsdatum",
            "Herstellernummer",
            "Werksnummer",
            "Fehlerhaft",
            "Fehlerhaft_Datum",
            "Fehlerhaft_Fahrleistung",
        ]

        missing = [
            col
            for col in required
            if col not in df.columns
        ]

        if missing:
            raise ValueError(
                f"{family}: missing columns {missing}"
            )

        clean = df[required].copy()

    elif f"{id_col}.x" in df.columns:
        clean = pd.DataFrame(index=df.index)

        id_candidates = [
            col
            for col in [
                f"{id_col}.x",
                f"{id_col}.y",
                id_col,
            ]
            if col in df.columns
        ]

        populated = (
            df[id_candidates]
            .notna()
            .sum(axis=1)
        )

        if (populated > 1).any():
            raise ValueError(
                f"{family}: "
                f"{int((populated > 1).sum())} rows "
                "have multiple component IDs"
            )

        if (populated == 0).any():
            raise ValueError(
                f"{family}: "
                f"{int((populated == 0).sum())} rows "
                "have no component ID"
            )

        for field in fields:
            candidates = [
                col
                for col in [
                    f"{field}.x",
                    f"{field}.y",
                    field,
                ]
                if col in df.columns
            ]

            if not candidates:
                continue

            result = df[candidates[0]].copy()

            for col in candidates[1:]:
                result = result.combine_first(
                    df[col]
                )

            clean[field] = result

        clean = clean.rename(
            columns={id_col: "ID"}
        )

    else:
        clean = pd.DataFrame(index=df.index)

        for field in fields:
            if field in df.columns:
                clean[field] = df[field]

        if (
            "Produktionsdatum" not in clean.columns
            and
            "Produktionsdatum_Origin_01011970"
            in df.columns
            and
            "origin" in df.columns
        ):
            days = pd.to_numeric(
                df[
                    "Produktionsdatum_Origin_01011970"
                ],
                errors="coerce"
            )

            origin = pd.to_datetime(
                df["origin"],
                format="%d-%m-%Y",
                errors="coerce"
            )

            clean["Produktionsdatum"] = (
                origin
                + pd.to_timedelta(
                    days,
                    unit="D"
                )
            )

        if id_col not in clean.columns:
            raise ValueError(
                f"{family}: could not find {id_col}. "
                f"Columns: {df.columns.tolist()}"
            )

        clean = clean.rename(
            columns={id_col: "ID"}
        )

    clean["Komponententyp"] = family

    clean["ID"] = (
        clean["ID"]
        .astype("string")
    )

    clean["Herstellernummer"] = (
        pd.to_numeric(
            clean["Herstellernummer"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    clean["Werksnummer"] = (
        pd.to_numeric(
            clean["Werksnummer"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    clean["Produktionsdatum"] = pd.to_datetime(
        clean["Produktionsdatum"],
        errors="coerce"
    )

    clean["Fehlerhaft_Datum"] = pd.to_datetime(
        clean["Fehlerhaft_Datum"],
        errors="coerce"
    )

    clean["Fehlerhaft"] = (
        pd.to_numeric(
            clean["Fehlerhaft"],
            errors="coerce"
        )
        .astype("Int8")
    )

    clean["Fehlerhaft_Fahrleistung"] = (
        pd.to_numeric(
            clean["Fehlerhaft_Fahrleistung"],
            errors="coerce"
        )
        .astype("Float64")
    )

    return clean[canonical_columns]


summary = []

for family in sorted(relevant_families):
    path = (
        KOMPONENTE_FOLDER
        / f"Komponente_{family}.csv"
    )

    if not path.exists():
        print(f"{family}: missing")

        summary.append({
            "component": family,
            "status": "missing",
        })

        continue

    sep = detect_separator(path)

    current_columns = pd.read_csv(
        path,
        sep=sep,
        nrows=0
    ).columns.tolist()

    if current_columns == canonical_columns:
        print(f"{family}: already clean")

        summary.append({
            "component": family,
            "status": "already clean",
        })

        continue

    temp_path = path.with_name(
        f".{path.name}.tmp"
    )

    if temp_path.exists():
        temp_path.unlink()

    total_rows = 0
    first_chunk = True
    missing_id = 0
    duplicate_id = 0
    faulty_without_date = 0
    nonfaulty_with_date = 0
    seen_ids = set()

    print(f"{family}: cleaning...")

    try:
        for raw in pd.read_csv(
            path,
            sep=sep,
            chunksize=50_000,
            dtype="string"
        ):
            clean = normalize_component_chunk(
                raw,
                family
            )

            missing_id += int(
                clean["ID"].isna().sum()
            )

            ids = clean["ID"].dropna()

            duplicate_id += int(
                ids.duplicated().sum()
            )

            unique_ids = ids.drop_duplicates()

            duplicate_id += sum(
                component_id in seen_ids
                for component_id in unique_ids
            )

            seen_ids.update(unique_ids)

            faulty_without_date += int(
                (
                    (clean["Fehlerhaft"] == 1)
                    &
                    clean["Fehlerhaft_Datum"].isna()
                ).sum()
            )

            nonfaulty_with_date += int(
                (
                    (clean["Fehlerhaft"] == 0)
                    &
                    clean["Fehlerhaft_Datum"].notna()
                ).sum()
            )

            clean.to_csv(
                temp_path,
                mode="a",
                header=first_chunk,
                index=False
            )

            first_chunk = False
            total_rows += len(clean)

            print(
                f"\r{family}: {total_rows:,} rows",
                end=""
            )

        if missing_id:
            raise ValueError(
                f"{family}: {missing_id} missing IDs"
            )

        if duplicate_id:
            raise ValueError(
                f"{family}: {duplicate_id} duplicate IDs"
            )

        if faulty_without_date:
            raise ValueError(
                f"{family}: "
                f"{faulty_without_date} defective rows "
                "without failure date"
            )

        if nonfaulty_with_date:
            raise ValueError(
                f"{family}: "
                f"{nonfaulty_with_date} non-defective rows "
                "with failure date"
            )

        temp_path.replace(path)

        print(
            f"\r{family}: finished — "
            f"{total_rows:,} rows"
        )

        summary.append({
            "component": family,
            "status": "cleaned",
            "rows": total_rows,
            "missing_ID": missing_id,
            "duplicate_ID": duplicate_id,
            "faulty_without_date":
                faulty_without_date,
            "nonfaulty_with_date":
                nonfaulty_with_date,
        })

    except Exception:
        if temp_path.exists():
            temp_path.unlink()

        raise


component_cleaning_summary = (
    pd.DataFrame(summary)
    .set_index("component")
)

display(component_cleaning_summary)


for family in sorted(relevant_families):
    path = (
        KOMPONENTE_FOLDER
        / f"Komponente_{family}.csv"
    )

    if not path.exists():
        continue

    columns = pd.read_csv(
        path,
        nrows=0
    ).columns.tolist()

    print(
        family,
        "OK" if columns == canonical_columns else "WRONG"
    )

K1BE1: cleaning...
K1BE1: finished — 1,192,630 rows
K1BE2: cleaning...
K1BE2: finished — 409,422 rows
K1DI1: cleaning...
K1DI1: finished — 1,192,630 rows
K1DI2: already clean
K2LE1: already clean
K2LE2: already clean
K2ST1: already clean
K2ST2: cleaning...
K2ST2: finished — 655,075 rows
K3AG1: cleaning...
K3AG1: finished — 477,052 rows
K3AG2: already clean
K3SG1: cleaning...
K3SG1: finished — 1,908,208 rows
K3SG2: cleaning...
K3SG2: finished — 655,075 rows
K4: cleaning...
K4: finished — 1,977,164 rows
K6: cleaning...
K6: finished — 512,354 rows


,status,rows,missing_ID,duplicate_ID,faulty_without_date,nonfaulty_with_date
component,,,,,,
K1BE1,cleaned,1192630.0,0.0,0.0,0.0,0.0
K1BE2,cleaned,409422.0,0.0,0.0,0.0,0.0
K1DI1,cleaned,1192630.0,0.0,0.0,0.0,0.0
K1DI2,already clean,NaN,NaN,NaN,NaN,NaN
K2LE1,already clean,NaN,NaN,NaN,NaN,NaN
K2LE2,already clean,NaN,NaN,NaN,NaN,NaN
K2ST1,already clean,NaN,NaN,NaN,NaN,NaN
K2ST2,cleaned,655075.0,0.0,0.0,0.0,0.0
K3AG1,cleaned,477052.0,0.0,0.0,0.0,0.0


K1BE1 OK
K1BE2 OK
K1DI1 OK
K1DI2 OK
K2LE1 OK
K2LE2 OK
K2ST1 OK
K2ST2 OK
K3AG1 OK
K3AG2 OK
K3SG1 OK
K3SG2 OK
K4 OK
K6 OK


check the untouched csv

In [40]:
canonical_part_columns = [
    "X1",
    "source_group",
    "ID",
    "Produktionsdatum",
    "Herstellernummer",
    "Werksnummer",
    "Fehlerhaft",
    "Fehlerhaft_Datum",
    "Fehlerhaft_Fahrleistung"
]

relevant_parts = [
    f"T{i:02d}"
    for i in list(range(1, 28)) + [30, 31, 32, 34, 35, 36, 37]
]

for part in relevant_parts:
    path = FOLDER / f"Einzelteil_{part}.csv"

    if not path.exists():
        raise FileNotFoundError(path)

    with path.open(
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:
        header = f.readline()

    sep = ";" if header.count(";") > header.count(",") else ","

    columns = pd.read_csv(
        path,
        sep=sep,
        nrows=0
    ).columns.tolist()

    if columns == canonical_part_columns:
        print(f"{part}: already canonical")
        continue

    temp_path = path.with_name(
        f".{path.name}.tmp"
    )

    if temp_path.exists():
        temp_path.unlink()

    first_chunk = True
    rows = 0
    missing_id = 0
    faulty_without_date = 0
    nonfaulty_with_date = 0

    try:
        for chunk in pd.read_csv(
            path,
            sep=sep,
            dtype="string",
            chunksize=100_000
        ):
            for column in chunk.columns:
                if column.startswith(
                    "Fehlerhaft_Fahrleistung"
                ):
                    chunk[column] = (
                        chunk[column]
                        .str.replace(
                            ",",
                            ".",
                            regex=False
                        )
                    )

            clean = normalize_einzelteil_chunk(
                chunk,
                part
            )

            rows += len(clean)

            missing_id += int(
                clean["ID"].isna().sum()
            )

            faulty_without_date += int(
                (
                    clean["Fehlerhaft"].eq(1)
                    & clean[
                        "Fehlerhaft_Datum"
                    ].isna()
                ).sum()
            )

            nonfaulty_with_date += int(
                (
                    clean["Fehlerhaft"].eq(0)
                    & clean[
                        "Fehlerhaft_Datum"
                    ].notna()
                ).sum()
            )

            clean.to_csv(
                temp_path,
                mode="a",
                header=first_chunk,
                index=False
            )

            first_chunk = False

        if missing_id:
            raise ValueError(
                f"{part}: {missing_id} missing IDs"
            )

        if faulty_without_date:
            raise ValueError(
                f"{part}: {faulty_without_date} faulty rows without failure date"
            )

        if nonfaulty_with_date:
            raise ValueError(
                f"{part}: {nonfaulty_with_date} non-faulty rows with failure date"
            )

        temp_path.replace(path)

        print(
            f"{part}: canonicalized — {rows:,} rows"
        )

    except Exception:
        if temp_path.exists():
            temp_path.unlink()
        raise

T01: already canonical
T02: already canonical
T03: already canonical
T04: canonicalized — 1,192,630 rows
T05: canonicalized — 1,192,630 rows
T06: canonicalized — 1,192,630 rows
T07: already canonical
T08: canonicalized — 409,422 rows
T09: already canonical
T10: canonicalized — 409,422 rows
T11: already canonical
T12: canonicalized — 1,908,208 rows
T13: canonicalized — 1,908,208 rows
T14: canonicalized — 477,052 rows
T15: canonicalized — 477,052 rows
T16: already canonical
T17: canonicalized — 655,075 rows
T18: canonicalized — 655,075 rows
T19: canonicalized — 163,769 rows
T20: already canonical
T21: canonicalized — 3,204,104 rows
T22: already canonical
T23: canonicalized — 1,908,208 rows
T24: already canonical
T25: canonicalized — 477,052 rows
T26: canonicalized — 655,075 rows
T27: already canonical
T30: canonicalized — 2,385,260 rows
T31: already canonical
T32: canonicalized — 1,977,164 rows
T34: already canonical
T35: already canonical
T36: already canonical
T37: canonicalized — 512,

In [41]:
verification = []

for part in relevant_parts:
    path = FOLDER / f"Einzelteil_{part}.csv"

    columns = pd.read_csv(
        path,
        nrows=0
    ).columns.tolist()

    sample = pd.read_csv(
        path,
        usecols=[
            "ID",
            "Werksnummer",
            "Fehlerhaft"
        ],
        nrows=1000
    )

    verification.append({
        "part": part,
        "schema_ok":
            columns == canonical_part_columns,
        "id_available":
            "ID" in columns,
        "werksnummer_available":
            "Werksnummer" in columns,
        "fehlerhaft_available":
            "Fehlerhaft" in columns,
        "sample_missing_id":
            sample["ID"].isna().sum()
    })

verification_df = (
    pd.DataFrame(verification)
    .set_index("part")
)

display(verification_df)

if not verification_df[
    [
        "schema_ok",
        "id_available",
        "werksnummer_available",
        "fehlerhaft_available"
    ]
].all().all():
    raise ValueError(
        "Not all Einzelteil files are canonical"
    )

if verification_df[
    "sample_missing_id"
].sum():
    raise ValueError(
        "Missing IDs found in Einzelteil samples"
    )

print("All relevant Einzelteil files are canonical.")

,schema_ok,id_available,werksnummer_available,fehlerhaft_available,sample_missing_id
part,,,,,
T01,True,True,True,True,0
T02,True,True,True,True,0
T03,True,True,True,True,0
T04,True,True,True,True,0
T05,True,True,True,True,0
T06,True,True,True,True,0
T07,True,True,True,True,0
T08,True,True,True,True,0
T09,True,True,True,True,0


All relevant Einzelteil files are canonical.
